# OR Intelligence · 01 · Data preparation (Member A)

Prepares the training data for the detector and hands it to Member B through Google Drive.
**Run this notebook first.** It does not need a GPU.

| Step | What happens | WBS |
|---|---|---|
| 1 | Settings (the only cell you edit) | — |
| 2 | Connect Google Drive | — |
| 3 | Unzip and check the raw dataset | 2.1.1 |
| 4 | Convert Pascal-VOC boxes → YOLO labels (6 instruments) + visual QA | 2.2.1 |
| 5 | Split **by video** into train / val / test | 2.2.2 |
| 6 | EDA: class balance, images per video, box sizes, co-occurrence | 2.4.1 |
| 7 | Save the ready dataset to Drive for Member B | hand-off |

**How to use:** upload the m2cai16-tool-locations zip to Google Drive once, set `RAW_ZIP` in step 1, then
`Runtime → Run all`.

Outputs in `MyDrive/or-intelligence/`: `data/yolo_dataset.zip`, `data/splits.json`, `results/eda/*`.
Copy `splits.json` to `data/splits/splits.json` in the repo, and the EDA figures to `docs/data/figures/`.

> Never commit the dataset, frames or videos to GitHub; they live in Drive (see `docs/data/LICENSES_AND_ETHICS.md`).

## 1 · Settings — the only cell you need to edit

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/or-intelligence"            # shared team folder
RAW_ZIP   = f"{DRIVE_DIR}/data/m2cai16-tool-locations.zip"      # the original dataset zip (upload once)
SEED      = 42
SPLIT     = {"train": 0.70, "val": 0.15, "test": 0.15}           # by video, never by frame

# the 6 instruments: must match contracts/classes.json in the repo
CLASSES = ["grasper", "bipolar", "hook", "scissors", "clipper", "irrigator"]
DROP    = {"specimenbag"}    # m2cai16 has a 7th class that we do not use

## 2 · Connect Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, json, random, re, shutil, zipfile, glob, hashlib, datetime
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from pathlib import Path

for sub in ["data", "runs", "weights", "results"]:
    os.makedirs(f"{DRIVE_DIR}/{sub}", exist_ok=True)
WORK = Path("/content/work")      # fast local disk; Drive keeps the results
WORK.mkdir(exist_ok=True)
print("Drive folder ready:", DRIVE_DIR)

## 3 · Unzip and check the raw dataset (WBS 2.1.1)

Compare the printed counts with the numbers on the dataset web page and note them in `docs/data/LICENSES_AND_ETHICS.md`.

In [ ]:
RAW = WORK / "raw"
if not RAW.exists():
    assert os.path.exists(RAW_ZIP), f"Upload the dataset zip to {RAW_ZIP} first"
    with zipfile.ZipFile(RAW_ZIP) as z:
        z.extractall(RAW)

xml_files = sorted(glob.glob(f"{RAW}/**/*.xml", recursive=True))
img_files = sorted(glob.glob(f"{RAW}/**/*.jpg", recursive=True) + glob.glob(f"{RAW}/**/*.png", recursive=True))
images_by_stem = {Path(p).stem: p for p in img_files}
print(f"XML annotations: {len(xml_files)}   images: {len(img_files)}")
print("example names:", [Path(p).stem for p in xml_files[:5]])
print("sha256 of the zip:", hashlib.sha256(open(RAW_ZIP, "rb").read()).hexdigest())

## 4 · Convert VOC boxes → YOLO labels (WBS 2.2.1)

Each box becomes `class cx cy w h` (0–1). SpecimenBag is dropped, boxes are clipped to the image, images without tools keep an empty label file.

In [ ]:
NAME_TO_ID = {n: i for i, n in enumerate(CLASSES)}
stats = Counter()

def convert(xml_path):
    root = ET.parse(xml_path).getroot()
    W = float(root.find("size/width").text); H = float(root.find("size/height").text)
    lines = []
    for obj in root.findall("object"):
        name = obj.find("name").text.strip().lower()
        if name in DROP:
            stats["dropped_specimenbag"] += 1; continue
        if name not in NAME_TO_ID:
            stats[f"unknown:{name}"] += 1; continue
        b = obj.find("bndbox")
        x1, y1, x2, y2 = (float(b.find(k).text) for k in ("xmin", "ymin", "xmax", "ymax"))
        x1, x2 = max(0, min(x1, W)), max(0, min(x2, W))
        y1, y2 = max(0, min(y1, H)), max(0, min(y2, H))
        if x2 <= x1 or y2 <= y1:
            stats["degenerate_box"] += 1; continue
        cx, cy, w, h = (x1 + x2) / 2 / W, (y1 + y2) / 2 / H, (x2 - x1) / W, (y2 - y1) / H
        lines.append(f"{NAME_TO_ID[name]} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
        stats[CLASSES[NAME_TO_ID[name]]] += 1
    return lines

labels = {}                       # image stem -> YOLO label lines
for x in xml_files:
    stem = Path(x).stem
    if stem not in images_by_stem:
        stats["xml_without_image"] += 1; continue
    try:
        labels[stem] = convert(x)
    except Exception:
        stats["malformed_xml"] += 1
stats["empty_images"] = sum(1 for v in labels.values() if not v)
print(f"converted {len(labels)} images")
print(dict(stats))

# sanity checks from the WBS acceptance criteria
for s, lines in labels.items():
    for line in lines:
        c, *xywh = line.split()
        assert 0 <= int(c) <= 5 and all(0 <= float(v) <= 1 for v in xywh), (s, line)
print("all class ids in 0..5 and all coordinates in [0, 1]")

In [ ]:
# Visual QA: 50 random images with their boxes -> look at them and note problems in docs/data/qa_notes.md
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
COLORS = ["#2563eb", "#7c3aed", "#dc2626", "#16a34a", "#ea580c", "#0891b2"]

def draw(stem):
    im = Image.open(images_by_stem[stem]).convert("RGB"); d = ImageDraw.Draw(im); W, H = im.size
    for line in labels[stem]:
        c, cx, cy, w, h = line.split(); c = int(c); cx, cy, w, h = map(float, (cx, cy, w, h))
        d.rectangle([(cx - w/2) * W, (cy - h/2) * H, (cx + w/2) * W, (cy + h/2) * H], outline=COLORS[c], width=3)
        d.text(((cx - w/2) * W + 3, (cy - h/2) * H + 3), CLASSES[c], fill=COLORS[c])
    return im

qa = random.Random(SEED).sample(sorted(labels), min(50, len(labels)))
os.makedirs(f"{DRIVE_DIR}/results/qa", exist_ok=True)
for s in qa:
    draw(s).save(f"{DRIVE_DIR}/results/qa/{s}.jpg")
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, s in zip(axes.flat, qa):
    ax.imshow(draw(s)); ax.set_title(s, fontsize=8); ax.axis("off")
plt.show()
print(f"{len(qa)} QA images saved to {DRIVE_DIR}/results/qa/")

## 5 · Split by video (WBS 2.2.2)

Frames from the same surgery never end up in two splits, so the test score measures unseen surgeries. **Check that the printed video IDs look right** (adjust `video_id()` if the file names use another pattern).

In [ ]:
def video_id(stem):
    # m2cai16 frames are named like <video>_<frame>; take everything before the last "_"
    m = re.match(r"(.+)_\d+$", stem)
    return m.group(1) if m else stem.split("_")[0]

videos = defaultdict(list)
for stem in labels:
    videos[video_id(stem)].append(stem)
print(f"{len(videos)} videos:", {v: len(s) for v, s in sorted(videos.items())})

def class_counts(stems):
    c = Counter()
    for s in stems:
        for line in labels[s]:
            c[CLASSES[int(line.split()[0])]] += 1
    return c

rng = random.Random(SEED)
vids = sorted(videos)
best = None
for attempt in range(500):     # first seeded shuffle where every split contains every class
    rng.shuffle(vids)
    n = len(vids); n_tr = round(n * SPLIT["train"]); n_va = max(1, round(n * SPLIT["val"]))
    split = {"train": vids[:n_tr], "val": vids[n_tr:n_tr + n_va], "test": vids[n_tr + n_va:]}
    if all(all(class_counts([s for v in split[k] for s in videos[v]])[c] > 0 for c in CLASSES) for k in split):
        best = {k: sorted(v) for k, v in split.items()}; break
assert best, "No split has all classes in every split: check video_id()"

all_v = best["train"] + best["val"] + best["test"]
assert len(all_v) == len(set(all_v)), "a video is in more than one split!"
print("leakage check passed:", {k: len(v) for k, v in best.items()}, "videos per split")

DS = WORK / "yolo"
if DS.exists(): shutil.rmtree(DS)
for k, vlist in best.items():
    (DS / "images" / k).mkdir(parents=True); (DS / "labels" / k).mkdir(parents=True)
    for v in vlist:
        for s in videos[v]:
            src = images_by_stem[s]
            shutil.copy(src, DS / "images" / k / Path(src).name)
            (DS / "labels" / k / f"{s}.txt").write_text("\n".join(labels[s]))
(DS / "dataset.yaml").write_text("path: .\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n"
                                 + "".join(f"  {i}: {n}\n" for i, n in enumerate(CLASSES)))
json.dump(best, open(f"{DRIVE_DIR}/data/splits.json", "w"), indent=2)
print("saved splits.json to Drive")

## 6 · EDA (WBS 2.4.1)

Four figures + a counts table, saved to `results/eda/`. Summarise them in `docs/data/EDA.md`, including recommendations for Member B (e.g. which class to oversample).

In [ ]:
import pandas as pd
EDA = f"{DRIVE_DIR}/results/eda"; os.makedirs(EDA, exist_ok=True)
split_of = {s: k for k in best for v in best[k] for s in videos[v]}

counts = pd.DataFrame({k: class_counts([s for v in best[k] for s in videos[v]]) for k in best}).reindex(CLASSES).fillna(0).astype(int)
counts["total"] = counts.sum(axis=1)
display(counts); counts.to_csv(f"{EDA}/boxes_per_class_per_split.csv")
print("imbalance ratio (max/min class):", round(counts["total"].max() / max(counts["total"].min(), 1), 2))

boxes = pd.DataFrame([{"split": split_of[s], "video": video_id(s), "cls": CLASSES[int(l.split()[0])],
                       "area": float(l.split()[3]) * float(l.split()[4])} for s in labels for l in labels[s]])
print("share of small boxes (< 1% of the image):", round((boxes.area < 0.01).mean(), 3))

fig, ax = plt.subplots(figsize=(8, 3)); counts.drop(columns="total").plot.bar(ax=ax, title="Boxes per class per split")
plt.tight_layout(); plt.savefig(f"{EDA}/boxes_per_class_per_split.png"); plt.show()

fig, ax = plt.subplots(figsize=(10, 3)); pd.Series({v: len(s) for v, s in sorted(videos.items())}).plot.bar(ax=ax, title="Images per video")
plt.tight_layout(); plt.savefig(f"{EDA}/images_per_video.png"); plt.show()

fig, ax = plt.subplots(figsize=(8, 3)); boxes.boxplot(column="area", by="cls", ax=ax); ax.set_title("Box area (fraction of image)")
plt.suptitle(""); plt.tight_layout(); plt.savefig(f"{EDA}/box_size_distribution.png"); plt.show()

co = pd.DataFrame(0, index=CLASSES, columns=CLASSES)
for s in labels:
    present = sorted({CLASSES[int(l.split()[0])] for l in labels[s]})
    for i in present:
        for j in present:
            co.loc[i, j] += 1
fig, ax = plt.subplots(figsize=(5, 4)); im = ax.imshow(co.values, cmap="Blues")
ax.set_xticks(range(6), CLASSES, rotation=45); ax.set_yticks(range(6), CLASSES); ax.set_title("Images where tools co-occur")
for i in range(6):
    for j in range(6):
        ax.text(j, i, co.values[i, j], ha="center", va="center", fontsize=8)
plt.tight_layout(); plt.savefig(f"{EDA}/class_cooccurrence.png"); plt.show()

## 7 · Save the ready dataset to Drive (hand-off to Member B)

In [ ]:
out_zip = f"{DRIVE_DIR}/data/yolo_dataset.zip"
shutil.make_archive(out_zip[:-4], "zip", DS)
print("saved", out_zip, f"({os.path.getsize(out_zip) / 1e6:.1f} MB)")
print("Tell Member B it is ready, and post this split summary in your PR:")
display(counts)

## Done — hand-off

| File in Drive | Who needs it | Repo location |
|---|---|---|
| `data/yolo_dataset.zip` | Member B (02_Model notebook) | Drive only, never committed |
| `data/splits.json` | Report | `data/splits/splits.json` |
| `results/eda/*.png`, `*.csv` | Member B, report, slides | `docs/data/figures/` |
| `results/qa/*.jpg` | You (QA notes) | notes in `docs/data/qa_notes.md` |